## Salinas BESS Monthly Invoice Calculator

This notebook follows the same workflow as the prior PPOA calculator: load inputs, validate them, run the contract calculations, write outputs, and inspect the results.

In [120]:
# Imports

import importlib
import sys
from pathlib import Path

import pandas as pd

PROJECT_DIR = Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# Import/reload local modules in dependency order.
module_names = [
    "classes",
    "calculations",
    "data_reader",
    "data_writer",
    "error_checks",
    "report",
    "compensation_calculator",
]

for module_name in module_names:
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])
    else:
        importlib.import_module(module_name)

calculations = sys.modules["calculations"]
compensation_calculator = sys.modules["compensation_calculator"]
data_reader = sys.modules["data_reader"]
data_writer = sys.modules["data_writer"]
error_checks = sys.modules["error_checks"]
report = sys.modules["report"]

FORCE_MAJEURE_WAITING_PERIOD_HOURS = calculations.FORCE_MAJEURE_WAITING_PERIOD_HOURS
GRID_SYSTEM_WAITING_PERIOD_HOURS = calculations.GRID_SYSTEM_WAITING_PERIOD_HOURS
SCHEDULED_MAINTENANCE_ALLOWANCE_HOURS = calculations.SCHEDULED_MAINTENANCE_ALLOWANCE_HOURS
calculate_monthly_results = compensation_calculator.calculate_monthly_results
load_bess_inputs = data_reader.load_bess_inputs
load_monthly_performance_guarantee_inputs = data_reader.load_monthly_performance_guarantee_inputs
load_performance_tests = data_reader.load_performance_tests
write_bess_monthly_results = data_writer.write_bess_monthly_results
input_validation = error_checks.input_validation
generate_bess_invoice_support_report = report.generate_bess_invoice_support_report

DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "output"

### Data Inputs and Preprocessing

In [121]:
# Project and filepaths

# Change this to "jobos" to run the same workflow for Jobos.
PROJECT_ID = "salinas"

PROJECT_DATA_DIR = DATA_DIR / PROJECT_ID
OUTPUT_DIR = PROJECT_DIR / "output" / PROJECT_ID

contract_values_csv = PROJECT_DATA_DIR / "bess_contract_values_template.csv"
yearly_inputs_csv = PROJECT_DATA_DIR / "bess_yearly_inputs_template.csv"
monthly_inputs_csv = PROJECT_DATA_DIR / "bess_monthly_inputs_template.csv"
monthly_performance_guarantee_csv = PROJECT_DATA_DIR / "Monthly_Performance_Guarantee.csv"
performance_tests_csv = PROJECT_DATA_DIR / "Performance_Tests.csv"

monthly_results_csv = OUTPUT_DIR / "bess_monthly_results.csv"
report_txt = OUTPUT_DIR / "report.txt"

In [122]:
# Input file validation

input_validation(
    contract_values_csv,
    yearly_inputs_csv,
    monthly_inputs_csv,
    monthly_performance_guarantee_csv,
    performance_tests_csv,
)

print("BESS input files passed validation.")

BESS input files passed validation.


In [123]:
# Load contract, yearly, monthly, performance guarantee, and performance test inputs

contract_values, yearly_inputs, monthly_inputs = load_bess_inputs(
    contract_values_csv,
    yearly_inputs_csv,
    monthly_inputs_csv,
)
monthly_performance_guarantee_inputs = load_monthly_performance_guarantee_inputs(
    monthly_performance_guarantee_csv
)
performance_tests = load_performance_tests(performance_tests_csv)

print(f"Loaded {len(contract_values)} contract value row(s).")
print(f"Loaded {len(yearly_inputs)} yearly input row(s).")
print(f"Loaded {len(monthly_inputs)} monthly input row(s).")
print(f"Loaded {len(monthly_performance_guarantee_inputs)} monthly performance guarantee row(s).")
print(f"Loaded {len(performance_tests)} performance test row(s).")

Loaded 25 contract value row(s).
Loaded 2 yearly input row(s).
Loaded 24 monthly input row(s).
Loaded 2 monthly performance guarantee row(s).
Loaded 1 performance test row(s).


In [124]:
# Preview source data

monthly_inputs_df = pd.read_csv(monthly_inputs_csv)
monthly_inputs_df.head()

,timestamp_month,agreement_year,Other_ADJ,BPHRS,POHRS,UNAVHRS,UNAVPRODHRS,GSE,PFM,IP
0,2026-01,1,0.0,744,8,2,1,3,0,0
1,2026-02,1,250.0,672,0,4,0,0,2,0
2,2026-03,1,0.0,744,16,0,3,5,0,1
3,2026-04,1,0.0,720,12,3,0,8,6,0
4,2026-05,1,125.0,744,0,5,1,12,0,0


### Calculations

In [125]:
# Calculate monthly compensation results

monthly_results = calculate_monthly_results(
    contract_values,
    yearly_inputs,
    monthly_inputs,
    performance_tests,
    monthly_performance_guarantee_inputs,
)

print(f"Calculated {len(monthly_results)} monthly result row(s).")

Calculated 24 monthly result row(s).


In [126]:
# Convert results to a review table

monthly_results_df = pd.DataFrame(
    [
        {
            "timestamp_month": result.timestamp_month,
            "agreement_year": result.agreement_year,
            "CPP": result.cpp,
            "MCC": result.mcc,
            "FA": result.fa,
            "FAA": result.faa,
            "PRA": result.pra,
            "MFP": result.mfp,
            "Other_ADJ": result.other_adj,
            "ALD": result.ald,
            "Actual_Efficiency": result.actual_efficiency,
            "ELD": result.eld,
            "ADJ_Total": result.adj_total,
            "MP": result.mp,
        }
        for result in monthly_results
    ]
)

monthly_results_df

,timestamp_month,agreement_year,CPP,MCC,FA,FAA,PRA,MFP,Other_ADJ,ALD,Actual_Efficiency,ELD,ADJ_Total,MP
0,2026-01,1,24896.00,100.0,0.995924,1.000000,0.995968,2.479561e+06,0.0,0.0,0.970000,0.000000,0.000000,2.479561e+06
1,2026-02,1,24896.00,100.0,0.994048,1.000000,0.997024,2.482190e+06,250.0,0.0,0.967969,27027.963392,27277.963392,2.454913e+06
2,2026-03,1,24896.00,100.0,0.995879,1.000000,0.991935,2.469523e+06,0.0,0.0,NaN,0.000000,0.000000,2.469523e+06
3,2026-04,1,24896.00,100.0,0.995763,1.000000,0.980556,2.441191e+06,0.0,0.0,NaN,0.000000,0.000000,2.441191e+06
4,2026-05,1,24896.00,100.0,0.991935,1.000000,0.983871,2.449445e+06,125.0,0.0,NaN,0.000000,125.000000,2.449320e+06
5,2026-06,1,24896.00,100.0,0.994253,1.000000,0.973611,2.423902e+06,0.0,0.0,NaN,0.000000,0.000000,2.423902e+06
6,2026-07,1,24896.00,100.0,0.998619,1.000000,0.947581,2.359097e+06,0.0,0.0,NaN,0.000000,0.000000,2.359097e+06
7,2026-08,1,24896.00,100.0,0.990463,1.000000,0.970430,2.415983e+06,300.0,0.0,NaN,0.000000,300.000000,2.415683e+06
8,2026-09,1,24896.00,100.0,0.995640,1.000000,0.944444,2.351289e+06,0.0,0.0,NaN,0.000000,0.000000,2.351289e+06
9,2026-10,1,24896.00,100.0,0.995968,1.000000,0.986559,2.456138e+06,0.0,0.0,NaN,0.000000,0.000000,2.456138e+06


### Annual Allowance Checks

In [127]:
# Review annual caps and waiting periods used by FA and PRA

allowance_summary_df = (
    monthly_inputs_df
    .groupby("agreement_year", as_index=False)
    .agg(
        POHRS=("POHRS", "sum"),
        GSE=("GSE", "sum"),
        PFM=("PFM", "sum"),
    )
)

allowance_summary_df["POHRS_allowance"] = SCHEDULED_MAINTENANCE_ALLOWANCE_HOURS
allowance_summary_df["GSE_waiting_period"] = GRID_SYSTEM_WAITING_PERIOD_HOURS
allowance_summary_df["PFM_waiting_period"] = FORCE_MAJEURE_WAITING_PERIOD_HOURS
allowance_summary_df["POHRS_over_allowance"] = (
    allowance_summary_df["POHRS"] - allowance_summary_df["POHRS_allowance"]
).clip(lower=0)
allowance_summary_df["GSE_over_waiting_period"] = (
    allowance_summary_df["GSE"] - allowance_summary_df["GSE_waiting_period"]
).clip(lower=0)
allowance_summary_df["PFM_over_waiting_period"] = (
    allowance_summary_df["PFM"] - allowance_summary_df["PFM_waiting_period"]
).clip(lower=0)

allowance_summary_df

,agreement_year,POHRS,GSE,PFM,POHRS_allowance,GSE_waiting_period,PFM_waiting_period,POHRS_over_allowance,GSE_over_waiting_period,PFM_over_waiting_period
0,1,168,87,94,160,80,720,8,7,0
1,2,176,72,766,160,80,720,16,0,46


### Write Output Files

In [128]:
# Write monthly result CSV

output_path = write_bess_monthly_results(monthly_results, output_dir=OUTPUT_DIR)
print(f"Wrote monthly results to {output_path}.")

Wrote monthly results to c:\Code\salinas-jobos-bess-invoice-calculator\output\salinas\bess_monthly_results.csv.


### Output Review

In [129]:
# Confirm the generated output file can be read back

written_results_df = pd.read_csv(monthly_results_csv)
written_results_df.head()

,timestamp_month,agreement_year,CPP,MCC,FA,FAA,PRA,MFP,Other_ADJ,ALD,CLD,Actual_Efficiency,ELD,ADJ_Total,MP
0,2026-01,1,24896.0,100.0,0.995924,1.0,0.995968,2.479561e+06,0.0,0.0,0.0,0.970000,0.000000,0.000000,2.479561e+06
1,2026-02,1,24896.0,100.0,0.994048,1.0,0.997024,2.482190e+06,250.0,0.0,0.0,0.967969,27027.963392,27277.963392,2.454913e+06
2,2026-03,1,24896.0,100.0,0.995879,1.0,0.991935,2.469523e+06,0.0,0.0,0.0,NaN,0.000000,0.000000,2.469523e+06
3,2026-04,1,24896.0,100.0,0.995763,1.0,0.980556,2.441191e+06,0.0,0.0,0.0,NaN,0.000000,0.000000,2.441191e+06
4,2026-05,1,24896.0,100.0,0.991935,1.0,0.983871,2.449445e+06,125.0,0.0,0.0,NaN,0.000000,125.000000,2.449320e+06


### Generate Invoice Support Report

This final cell calls `generate_bess_invoice_support_report()` from `report.py`.

In [130]:
# Generate a Section 10.1-style invoice support report using report.py

report_text = generate_bess_invoice_support_report(written_results_df, report_txt)
print(f"Wrote invoice support report to {report_txt}.")
print("\n".join(report_text.splitlines()[:45]))

Wrote invoice support report to c:\Code\salinas-jobos-bess-invoice-calculator\output\salinas\report.txt.
BESS MONTHLY INVOICE SUPPORT REPORT
Source: Section 10.1 Payment & Billings invoice-content requirements
Generated: 2026-06-08 13:56:54

----------------------------------------------------------------------------------------
Billing Period: 2026-01    Agreement Year: 1
----------------------------------------------------------------------------------------

Monthly Fixed Payment
  CPP:  $24,896.00
  MCC:  100.000000 MW
  FA:   99.59%
  FAA:  100.00%
  PRA:  99.60%
  MFP:  $2,479,561.29

Other Payment Adjustments / Credits Owing to PREPA
  Other_ADJ: $0.00
  ALD:       $0.00
  CLD:       $0.00
  Actual Efficiency: 97.00%
  ELD:       $0.00
  ADJ_Total: $0.00

Monthly Payment
  MP = MFP - ADJ_Total
  MP: $2,479,561.29

Not Yet Populated / Itemized Separately
  Discharge Energy and Charge Energy detail: see monthly performance input file
  Ancillary Services: not provided in current i